# Compare M1 with Coefficient Baselines

This notebook runs Step 16 of the thesis prototype:

$$\delta_i \rightarrow R \rightarrow \lambda = f(p, R) \rightarrow \theta(\lambda)$$

It compares M1 with uniform and direct-preference coefficients. If the fixed-grid summary is available, the notebook also uses its best utility as a reference.

It uses already trained helpful and harmless GPT-2 LoRA adapters, generates a small set of responses, and applies lightweight heuristic proxy scores. For faster execution, select a GPU runtime in Colab before starting.

## 1. Clone or update the repository

This cell starts in `/content`. If `/content/master-thesis/.git` exists, it updates the existing repository. Otherwise, it clones a fresh copy. If a non-Git folder already uses that path, the cell stops instead of creating a nested folder.

In [ ]:
%cd /content

from pathlib import Path
import subprocess

repo_path = Path("/content/master-thesis")

if (repo_path / ".git").is_dir():
    print("Repository already exists. Pulling the latest changes...")
    subprocess.run(["git", "-C", str(repo_path), "pull"], check=True)
elif repo_path.exists():
    raise RuntimeError(
        "/content/master-thesis exists but is not a Git repository. "
        "Rename or remove that folder, then run this cell again."
    )
else:
    print("Cloning the repository...")
    subprocess.run(
        ["git", "clone", "https://github.com/NZhang137/master-thesis.git"],
        check=True,
    )

%cd /content/master-thesis

## 2. Inspect the repository

These commands confirm that Colab is in the repository root and show the scripts and result files currently available.

In [ ]:
!pwd
!ls
!ls scripts
!ls results

## 3. Install dependencies

The comparison loads GPT-2 and the two PEFT adapters. Pandas is used to preview the small result tables.

`torchao` is not needed for this GPT-2 + PEFT prototype. Colab may contain an old incompatible version, so the installation cell removes it first. Colab's existing PyTorch installation is kept.

In [ ]:
!pip uninstall -y torchao
!pip install -q -U pandas numpy transformers datasets peft accelerate safetensors

After running the installation cell, select **Runtime > Restart session** if `torchao` was previously imported or if you already saw a `torchao` compatibility error. Then rerun the repository and installation cells before continuing.

## 4. Check the GPU

The script can run on CPU, but a Colab GPU makes repeated GPT-2 generation much faster.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. In Colab, choose Runtime > Change runtime type > GPU.")

## 5. Check the required result files

The workflow expects:

- `results/relationship_matrix.csv` from Step 14
- `results/m1_coefficients.csv` from Step 15
- `results/lambda_sweep_summary.csv` from Step 13

The comparison script computes its M1 candidates directly from the relationship matrix. The Step 15 CSV confirms the standalone M1 computation was completed. The fixed-grid summary supplies the best-grid reference; if it is missing, the script can still compare uniform, direct, and M1, but its grid-reference columns remain empty.

In [ ]:
from pathlib import Path

required_results = {
    "results/relationship_matrix.csv": (
        "Step 14 is required first: python scripts/compute_relationship_matrix.py"
    ),
    "results/m1_coefficients.csv": (
        "Step 15 is required first: python scripts/compute_m1_coefficients.py"
    ),
    "results/lambda_sweep_summary.csv": (
        "Step 13 is required first: python scripts/evaluate_lambda_sweep.py"
    ),
}

for path_text, missing_message in required_results.items():
    path = Path(path_text)
    if path.is_file():
        print(f"Found:   {path_text}")
    else:
        print(f"Missing: {path_text}")
        print(f"         {missing_message}")

## 6. Check whether the adapters exist

Both trained adapters must be available locally:

- `adapters/gpt2-helpful-adapter`
- `adapters/gpt2-harmless-adapter`

If both folders exist, skip the optional upload cells.

In [ ]:
helpful_path = Path("adapters/gpt2-helpful-adapter")
harmless_path = Path("adapters/gpt2-harmless-adapter")

print("Helpful adapter exists:", helpful_path.is_dir())
print("Harmless adapter exists:", harmless_path.is_dir())
!ls adapters || echo "No adapters folder found."

## 7. Upload `adapters.zip` if needed

Run the next two cells only if the adapter folders are missing. Select your local `adapters.zip` backup when prompted.

`adapters.zip` is a local backup only. It and the extracted adapter weights must not be committed to GitHub.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
!if [ -f adapters.zip ]; then unzip -o adapters.zip; else echo "No adapters.zip found, skipping unzip."; fi
!ls adapters || echo "No adapters folder found."

## 8. Check the adapter files

The checker verifies that both folders contain the PEFT configuration and adapter-weight files expected by the comparison script.

In [ ]:
!python scripts/check_adapters.py

## 9. Run the M1 baseline comparison

This command evaluates uniform, direct-preference, and M1 coefficients on the shared three-prompt set. The default M1 correction strength is `tau=1.0`.

In [ ]:
!python scripts/compare_m1_to_baselines.py

## 10. Inspect the outputs

The script creates:

- `results/m1_baseline_generations.csv`: response-level generations and proxy scores
- `results/m1_baseline_comparison.csv`: aggregated method comparison for each preference

In [ ]:
!ls results
!cat results/m1_baseline_comparison.csv

In [ ]:
import pandas as pd

generations_df = pd.read_csv("results/m1_baseline_generations.csv")
comparison_df = pd.read_csv("results/m1_baseline_comparison.csv")

print("Generation preview:")
display(generations_df.head())

print("Comparison table:")
display(comparison_df)

## 11. Understand the comparison

- **Uniform:** $\lambda = [0.5, 0.5]$ for every preference.
- **Direct preference:** $\lambda = p$.
- **M1:** relationship-softmax correction $\lambda = f(p, R)$.
- **Best grid:** the highest preference-weighted utility found in the earlier fixed lambda sweep, if its summary file is available.

`utility` combines the mean helpfulness and harmlessness proxies using the selected preference vector. `gap_to_best_grid_if_available` is candidate utility minus the stored best-grid utility, so a positive value means the candidate scored above that grid reference.

All helpfulness, harmlessness, and utility values here are lightweight heuristic prototype scores. They are not final reward-model scores and should not be interpreted as a final model-quality result.

## 12. Git safety check

It is okay to commit the small CSV result files under `results/`.

Do **not** commit:

- `adapters/`
- `adapters.zip`
- `.safetensors` or `.bin` files
- checkpoints or full model files

In [ ]:
!git status

## What this notebook establishes

This notebook provides a small, reproducible infrastructure and heuristic-evaluation comparison of M1 against simple coefficient baselines inside the fixed merging family.